# 🏆 Leaderboard · Task 1 · MNIST digits

Starter for Task 1 of the [Leaderboard](Leaderboard.md): classify handwritten digits, scored by **test accuracy** (higher is better).

| | |
|---|---|
| **Data** | MNIST, 70 000 grey-scale 28×28 digits (Keras mirror `mnist.npz`, 11 MB) |
| **Split** | the official 60 000 train / 10 000 test. Validation = the **last 10 000 training images** |
| **Metric** | accuracy on the 10 000 test images |
| **Baseline** | logistic regression on pixels / 255 (scikit-learn defaults, `max_iter=100`): **0.9257** |
| **Rules** | test labels are used only by `mnist_accuracy`, once per idea · no model pre-trained on MNIST · augmentation, ensembles and any training time are fine |


**How to use this notebook**
1. Run everything once as delivered: it downloads the data, checks the split is the official one, and reproduces the baseline.
2. Write your model in **`my_model`** (section 4) and set `METHOD` to a short description.
3. Iterate on the **validation** score (section 5) as often as you like.
4. When you are happy, run section 6 **once**: it trains on the full training data, scores the test set, and prints a row for [Leaderboard](Leaderboard.md).
5. Save a copy of the notebook for your attempt if you want the row's link to point to the exact code (*File → Save as*, e.g. `Leaderboard - MNIST - small CNN.ipynb`).

Cells print ✅ / ❌ / ⏳ instead of raising, so the notebook always runs top to bottom. Saved without outputs.

In [ ]:
import datetime, hashlib, pathlib, re, time, urllib.request
import numpy as np

DATA = pathlib.Path("leaderboard-data")   # downloads are cached here (git-ignored)
DATA.mkdir(exist_ok=True)

def check(ok, msg_ok, msg_bad):
    print(("✅ " + msg_ok) if ok else ("❌ " + msg_bad))

def download(url, name):
    f = DATA / name
    if not f.exists():
        print("downloading", url)
        urllib.request.urlretrieve(url, f)
    return f

def leaderboard_best(task, higher_is_better, path="Leaderboard.md"):
    """Best score logged so far in the Leaderboard.md section whose heading contains `task`."""
    p = pathlib.Path(path)
    if not p.exists():
        return None
    best = None
    for sec in re.split(r"^## ", p.read_text(encoding="utf-8"), flags=re.M)[1:]:
        if task not in sec.splitlines()[0]:
            continue
        for line in sec.splitlines():
            cells = [c.strip() for c in line.strip().strip("|").split("|")]
            if len(cells) < 3 or not re.fullmatch(r"\d{4}-\d{2}-\d{2}", cells[0]):
                continue
            try:
                score = float(cells[1])
            except ValueError:
                continue
            if best is None or (score > best[0] if higher_is_better else score < best[0]):
                best = (score, cells[2])
    return best

def report(task, score, method, baseline, higher_is_better, notebook, digits=4):
    """Compare a test score with the baseline and your best so far, and print a row for Leaderboard.md."""
    better = (lambda a, b: a > b) if higher_is_better else (lambda a, b: a < b)
    print(f"Test score: {score:.{digits}f}")
    check(better(score, baseline), f"beats the baseline ({baseline})", f"does not beat the baseline ({baseline}) yet")
    best = leaderboard_best(task, higher_is_better)
    if best:
        check(better(score, best[0]), f"new personal best! Previous best: {best[0]} ({best[1]})",
              f"not a new best. Current best: {best[0]} ({best[1]})")
    print("\nPaste this row at the bottom of the table in Leaderboard.md:")
    print(f"| {datetime.date.today()} | {score:.{digits}f} | {method} | [notebook]({notebook.replace(' ', '%20')}) |")

f = download("https://storage.googleapis.com/tensorflow/tf-keras-datasets/mnist.npz", "mnist.npz")
with np.load(f) as d:
    X_train_full, y_train_full, X_test, y_test = d["x_train"], d["y_train"], d["x_test"], d["y_test"]
print(X_train_full.shape, X_test.shape, X_train_full.dtype)

## 1. The fixed split
The fingerprint makes sure your copy of the data is byte-for-byte the one every leaderboard score was computed on.

In [ ]:
X_train, y_train = X_train_full[:50_000], y_train_full[:50_000]   # fit here while developing
X_val, y_val = X_train_full[50_000:], y_train_full[50_000:]       # ...and compare ideas here

fp = hashlib.sha256(X_test.tobytes() + y_test.tobytes()).hexdigest()[:12]
check(fp == "c3f9adf9c66e" and len(X_train) == 50_000 and len(X_val) == 10_000,
      "data and split are the official ones", f"unexpected data (fingerprint {fp}): scores are not comparable")

## 2. The official metric
Every leaderboard score comes from this function. It refuses predictions of the wrong shape instead of silently broadcasting.

In [ ]:
def accuracy(y_true, y_pred):
    y_pred = np.asarray(y_pred).ravel()
    assert y_pred.shape == y_true.shape, f"expected {y_true.shape[0]} predictions, got {y_pred.shape[0]}"
    return float((y_pred == y_true).mean())

def mnist_accuracy(y_pred):
    """Official Task 1 metric: accuracy on the 10 000 MNIST test images."""
    return accuracy(y_test, y_pred)

check(mnist_accuracy(y_test) == 1.0 and mnist_accuracy(np.zeros(10_000, int)) == np.mean(y_test == 0),
      "metric self-test passed", "metric self-test failed")

## 3. Baseline
Logistic regression on raw pixels, trained on all 60 000 training images. Takes about 15 s on a laptop CPU.
(The solver stops at `max_iter=100` before full convergence; that is part of the baseline definition. Expect the last digit to wobble across machines.)

In [ ]:
import warnings
from sklearn.linear_model import LogisticRegression

BASELINE = 0.9257
t = time.time()
with warnings.catch_warnings():
    warnings.simplefilter("ignore")   # ConvergenceWarning: expected, see above
    base = LogisticRegression(max_iter=100).fit(X_train_full.reshape(60_000, -1) / 255, y_train_full)
base_acc = mnist_accuracy(base.predict(X_test.reshape(10_000, -1) / 255))
print(f"baseline test accuracy {base_acc:.4f} ({time.time() - t:.0f} s)")
check(abs(base_acc - BASELINE) < 0.002, f"baseline reproduced (≈ {BASELINE})", f"baseline differs from {BASELINE}")

## 4. Your model ✏️
`my_model(X_fit, y_fit, X_eval)` gets uint8 images of shape `(n, 28, 28)` and must return one predicted label per row of `X_eval`.
Return `None` until you have something; the later cells then print ⏳.

In [ ]:
METHOD = "my model"   # short description for the leaderboard row, e.g. "CNN 2×conv, 5 epochs, augmentation"

def my_model(X_fit, y_fit, X_eval):
    # TODO: train on (X_fit, y_fit) and return predictions for X_eval
    return None

<details>
<summary><b>Example submission</b> (shows the interface; try your own idea first)</summary>

k-nearest neighbours on raw pixels. Scores **0.9705** test accuracy, about 10 s.

```python
from sklearn.neighbors import KNeighborsClassifier

METHOD = "kNN k=3 on raw pixels"

def my_model(X_fit, y_fit, X_eval):
    knn = KNeighborsClassifier(n_neighbors=3, n_jobs=-1).fit(X_fit.reshape(len(X_fit), -1) / 255, y_fit)
    return knn.predict(X_eval.reshape(len(X_eval), -1) / 255)
```

</details>

**Ideas to try:** tune k and the distance (validation!), PCA before kNN, an SVM with an RBF kernel, a small CNN in PyTorch, data augmentation (shifts, small rotations), an ensemble of a few CNNs.
See [[k-Nearest Neighbors]], [[Support Vector Machines]], [[CNN]], [[Training Tricks]].

## 5. Validation score (use as often as you like)

In [ ]:
t = time.time()
pred_val = my_model(X_train, y_train, X_val)
if pred_val is None:
    print("⏳ my_model returns None: write your model in section 4")
else:
    print(f"validation accuracy {accuracy(y_val, pred_val):.4f} ({time.time() - t:.0f} s)")

## 6. Test score (once per idea)
Refits on all 60 000 training images, scores the test set and compares with the baseline and with your best row in `Leaderboard.md`.

In [ ]:
pred_test = my_model(X_train_full, y_train_full, X_test)
if pred_test is None:
    print("⏳ my_model returns None: nothing to submit yet")
else:
    report("MNIST", mnist_accuracy(pred_test), METHOD, BASELINE, higher_is_better=True,
           notebook="Leaderboard - MNIST.ipynb")